# parameter-wrap-around-tensor — faded example 2: complete the WrapParam conversion

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `parameter-wrap-around-tensor`. Running the beacon reports progress on the `Backprop: Parameter wrap around Tensor` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Parameter wrap around Tensor` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`parameter-wrap-around-tensor`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "parameter-wrap-around-tensor"
DD_SUBTOPIC = "Backprop: Parameter wrap around Tensor"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

Repairing a HAS-A Parameter means building an IS-A `IsAParam` from its stored tensor while preserving the backing array. Other items pass through unchanged.

## Faded exercise 2

Complete `fix_params` so each WrapParam becomes an IsAParam built from its `.tensor`. Fill in the conversion branch.

**Fill in:** the IsAParam(p.tensor) conversion for WrapParam items

In [ ]:
import numpy as np

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = np.asarray(array, dtype=float)
        self.requires_grad = requires_grad

class WrapParam:
    def __init__(self, tensor):
        self.tensor = tensor
        self.requires_grad = True

class IsAParam(MiniTensor):
    def __init__(self, array):
        super().__init__(array, requires_grad=True)

def fix_params(things):
    out = []
    for p in things:
        if isinstance(p, WrapParam):
            converted = IsAParam(p.tensor)
            out.append(converted)
        else:
            out.append(p)
    return out

fixed = fix_params([WrapParam(np.array([1.0]))])
print(isinstance(fixed[0], MiniTensor))


def _test():
    arr = np.array([3.0, 4.0])
    bag = [WrapParam(arr), IsAParam([9.0])]
    fixed = fix_params(bag)
    # WrapParam converted to a MiniTensor-typed param
    assert isinstance(fixed[0], IsAParam)
    assert isinstance(fixed[0], MiniTensor)
    assert np.allclose(fixed[0].array, [3.0, 4.0])
    # the already-correct one is untouched
    assert fixed[1] is bag[1]
    # input list not mutated
    assert isinstance(bag[0], WrapParam)


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import numpy as np

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = np.asarray(array, dtype=float)
        self.requires_grad = requires_grad

class WrapParam:
    def __init__(self, tensor):
        self.tensor = tensor
        self.requires_grad = True

class IsAParam(MiniTensor):
    def __init__(self, array):
        super().__init__(array, requires_grad=True)

def fix_params(things):
    out = []
    for p in things:
        if isinstance(p, WrapParam):
            converted = IsAParam(p.tensor)
            out.append(converted)
        else:
            out.append(p)
    return out

fixed = fix_params([WrapParam(np.array([1.0]))])
print(isinstance(fixed[0], MiniTensor))
```
</details>